> ⚠️ **Tratamento de valores nulos**

Não adianta deletar todos os registros que contenham `NULL`, pois estes fatos existiram e removê-los poderia distorcer a análise final. Por isso, optei por **manter os nulos**.

Em casos onde a imputação faz sentido, as estratégias possíveis são:

| Estratégia | Quando usar |
|---|---|
| Deletar o registro | Campo nulo inviabiliza qualquer vínculo ou análise |
| Preencher pela média | Campos numéricos contínuos com ausência aleatória |
| Preencher pela moda | Campos categóricos com valor predominante conhecido |

| Campo | Tabela | Recomendação |
|---|---|---|
| `canal_id` | `dim_vendedores` | Mantém NULL — não inventar vínculo |
| `regional_code` | `dim_vendedores` | Mantém NULL — não há como inferir |
| `status` | `dim_vendedores` | Mantém NULL — ausência é informação |
| `delivery_status` | `tb_entrega` | Mantém NULL — indica entrega sem status |
| `severity` | `tb_atendimentos` | Mantém NULL — ocorrência sem classificação |
| `cost` | `tb_entrega` | Mantém NULL — não imputar valor financeiro |
| `carrier_name` | `tb_entrega` | Mantém NULL — transportadora desconhecida |

In [0]:
%sql
-- Não existem vendedores para o canal 06
select * 
FROM workspace.silver.dim_vendedores 
where canal_id in ('CH06');
delete from workspace.silver.dim_canais where id_canal = 'CH06';

In [0]:
%sql
-- Status Inativo e Nulo dos vendedores corresponde a quase 50% das vendas, então vou considerar estas vendas, já que o vendedor pode ter ido embora mas sua venda continua na análise. Mesma coisa para canal e região, a venda pode ter ocorrido para uma região e não ter um canal definido ou vice-versa.
select count(*) 
from workspace.silver.tb_pedidos_cabecalho
where seller_id in ( 
select distinct seller_id 
FROM workspace.silver.dim_vendedores 
where status = 'Inativo' or status is null)


In [0]:
%sql
SELECT
  'pedidos_sem_vendedor'     AS verificacao, COUNT(*) AS total 
   FROM workspace.silver.tb_pedidos_cabecalho p 
   LEFT JOIN workspace.silver.dim_vendedores v ON p.seller_id = v.seller_id WHERE v.seller_id IS NULL
UNION ALL
SELECT 'pedidos_sem_cliente',                COUNT(*) 
FROM workspace.silver.tb_pedidos_cabecalho p 
LEFT JOIN workspace.silver.dim_clientes c ON p.customer_code = c.customer_id WHERE c.customer_id IS NULL
UNION ALL
SELECT 'itens_sem_pedido',                   COUNT(*) 
FROM workspace.silver.tb_pedidos_itens i 
LEFT JOIN workspace.silver.tb_pedidos_cabecalho p ON i.order_id = p.order_id WHERE p.order_id IS NULL
UNION ALL
SELECT 'itens_sem_produto',                  COUNT(*) FROM workspace.silver.tb_pedidos_itens i LEFT JOIN workspace.silver.dim_produto pr ON i.product_code = pr.product_id WHERE pr.product_id IS NULL
UNION ALL
SELECT 'entregas_sem_pedido',                COUNT(*) FROM workspace.silver.tb_entrega e LEFT JOIN workspace.silver.tb_pedidos_cabecalho p ON e.order_ref = p.order_id WHERE p.order_id IS NULL
UNION ALL
SELECT 'atendimentos_sem_pedido',            COUNT(*) FROM workspace.silver.tb_atendimentos a LEFT JOIN workspace.silver.tb_pedidos_cabecalho p ON a.order_id = p.order_id WHERE p.order_id IS NULL
UNION ALL
SELECT 'vendedores_sem_canal',               COUNT(*) FROM workspace.silver.dim_vendedores v LEFT JOIN workspace.silver.dim_canais c ON v.canal_id = c.id_canal WHERE c.id_canal IS NULL AND v.canal_id IS NOT NULL
UNION ALL
SELECT 'vendedores_sem_regiao',              COUNT(*) FROM workspace.silver.dim_vendedores v LEFT JOIN workspace.silver.dim_regioes r ON v.regional_code = r.regional_code WHERE r.regional_code IS NULL AND v.regional_code IS NOT NULL
ORDER BY total DESC;